In [11]:
from dataclasses import dataclass
from pathlib import Path

import klayout.db as kdb


GDS_PATH = Path("../puzzle.gds")
PDK_PATH = Path("/Users/korotkingleb/.ciel/sky130A")

REFERENCE_GDS = (
    PDK_PATH
    / "libs.ref"
    / "sky130_fd_sc_hd"
    / "gds"
    / "sky130_fd_sc_hd.gds"
)

MUX2 = "sky130_fd_sc_hd__mux2_1"
DFFR = "sky130_fd_sc_hd__dfrtp_2"

# SKY130 conductor/via stack from the installed KLayout technology.
# LI and above are enough to connect standard-cell signal pins.
CONDUCTOR_STACK = [
    ("li", 67, 20),
    ("mcon", 67, 44),
    ("met1", 68, 20),
    ("via1", 68, 44),
    ("met2", 69, 20),
    ("via2", 69, 44),
    ("met3", 70, 20),
    ("via3", 70, 44),
    ("met4", 71, 20),
    ("via4", 71, 44),
    ("met5", 72, 20),
]

# SKY130 standard-cell signal pin names are labels on LI.
LI_LABEL = (67, 5)


@dataclass
class RecognizedCell:
    instance_id: int
    cell_type: str
    transform: kdb.Trans
    # A pin may have multiple GDS labels, but they should resolve to the
    # same extracted electrical net.
    pin_nets: dict[str, set[int]]


@dataclass
class RegisterStage:
    mux: RecognizedCell
    dff: RecognizedCell


def load_pdk_cell_names() -> set[str]:
    """Return the cell masters present in the official PDK GDS."""
    reference = kdb.Layout()
    reference.read(str(REFERENCE_GDS))
    return {cell.name for cell in reference.each_cell()}


def extract_conductor_nets(
    layout: kdb.Layout,
    top: kdb.Cell,
) -> kdb.LayoutToNetlist:
    """Extract flat metal connectivity directly from GDS polygons."""
    l2n = kdb.LayoutToNetlist(top.name, layout.dbu)
    layers: dict[str, kdb.Region] = {}

    for name, gds_layer, datatype in CONDUCTOR_STACK:
        layer_index = layout.layer(gds_layer, datatype)
        region = kdb.Region(top.begin_shapes_rec(layer_index))
        l2n.register(region, name)
        l2n.connect(region)
        layers[name] = region

    # Connect each adjacent conductor/via/conductor item in the PDK stack.
    for lower, upper in zip(CONDUCTOR_STACK, CONDUCTOR_STACK[1:]):
        l2n.connect(layers[lower[0]], layers[upper[0]])

    l2n.extract_netlist()
    return l2n


def extract_pin_nets(
    layout: kdb.Layout,
    instance: kdb.Instance,
    l2n: kdb.LayoutToNetlist,
) -> dict[str, set[int]]:
    """Transform GDS pin labels and probe the extracted net at each label."""
    label_layer = layout.layer(*LI_LABEL)
    li_region = l2n.polygons_by_name("li")
    result: dict[str, set[int]] = {}

    for shape in instance.cell.shapes(label_layer).each():
        if not shape.is_text():
            continue

        text = shape.text
        displacement = text.trans.disp
        local_point = kdb.Point(displacement.x, displacement.y)
        top_point = instance.trans * local_point
        net = l2n.probe_net(li_region, top_point)

        if net is not None:
            result.setdefault(text.string, set()).add(net.cluster_id)

    return result


def pins_connected(
    first: RecognizedCell,
    first_pin: str,
    second: RecognizedCell,
    second_pin: str,
) -> bool:
    """Test connectivity using net IDs extracted from GDS geometry."""
    return bool(
        first.pin_nets.get(first_pin, set())
        & second.pin_nets.get(second_pin, set())
    )


def is_stage(mux: RecognizedCell, dff: RecognizedCell) -> bool:
    """Recognize one enable-controlled shift-register stage."""
    return (
        mux.cell_type == MUX2
        and dff.cell_type == DFFR
        and pins_connected(mux, "X", dff, "D")
        and pins_connected(mux, "A0", dff, "Q")
    )


def follows(previous: RegisterStage, following: RegisterStage) -> bool:
    """Return True when the following stage shifts from the previous Q."""
    return pins_connected(previous.dff, "Q", following.mux, "A1")


def share_pin_net(
    stages: list[RegisterStage],
    part: str,
    pin: str,
) -> bool:
    """Check that every selected MUX or DFF pin shares one physical net."""
    common: set[int] | None = None

    for stage in stages:
        cell = stage.mux if part == "mux" else stage.dff
        nets = cell.pin_nets.get(pin, set())
        common = set(nets) if common is None else common & nets

    return bool(common)


def find_shift_registers(stages: list[RegisterStage]) -> list[list[RegisterStage]]:
    """Build ordered chains and verify shared enable, clock, and reset nets."""
    successors: dict[int, list[int]] = {index: [] for index in range(len(stages))}
    predecessors: dict[int, list[int]] = {index: [] for index in range(len(stages))}

    for previous_index, previous in enumerate(stages):
        for following_index, following in enumerate(stages):
            if previous_index == following_index:
                continue
            if follows(previous, following):
                successors[previous_index].append(following_index)
                predecessors[following_index].append(previous_index)

    chains: list[list[RegisterStage]] = []

    for root in (index for index in predecessors if not predecessors[index]):
        indexes: list[int] = []
        current = root

        while current not in indexes:
            indexes.append(current)
            if len(successors[current]) != 1:
                break
            current = successors[current][0]

        chain = [stages[index] for index in indexes]

        # A single MUX+DFF is only a register stage, not enough evidence for
        # a shift register. A chain must also share its control signals.
        if (
            len(chain) >= 2
            and share_pin_net(chain, "mux", "S")
            and share_pin_net(chain, "dff", "CLK")
            and share_pin_net(chain, "dff", "RESET_B")
        ):
            chains.append(chain)

    # Make output deterministic: upper chains appear before lower chains.
    return sorted(
        chains,
        key=lambda chain: sum(stage.dff.transform.disp.y for stage in chain),
        reverse=True,
    )


def main() -> None:
    if not GDS_PATH.is_file():
        raise FileNotFoundError(f"GDS not found: {GDS_PATH}")
    if not REFERENCE_GDS.is_file():
        raise FileNotFoundError(f"PDK reference GDS not found: {REFERENCE_GDS}")

    layout = kdb.Layout()
    layout.read(str(GDS_PATH))
    top = layout.top_cell()

    known_pdk_cells = load_pdk_cell_names()
    l2n = extract_conductor_nets(layout, top)

    recognized: list[RecognizedCell] = []

    for instance_id, instance in enumerate(top.each_inst()):
        cell_type = instance.cell.name

        recognized.append(
            RecognizedCell(
                instance_id=instance_id,
                cell_type=cell_type,
                transform=instance.trans,
                pin_nets=extract_pin_nets(layout, instance, l2n),
            )
        )
    print(recognized)
    '''
    muxes = [cell for cell in recognized if cell.cell_type == MUX2]
    dffs = [cell for cell in recognized if cell.cell_type == DFFR]
    stages = [
        RegisterStage(mux, dff)
        for mux in muxes
        for dff in dffs
        if is_stage(mux, dff)
    ]
    shift_registers = find_shift_registers(stages)

    print(f"Top cell: {top.name}")
    print(f"PDK-recognized candidate cells: {len(recognized)}")
    print(f"MUX+DFF register stages: {len(stages)}")
    print(f"Shift-register chains: {len(shift_registers)}")

    for register_number, chain in enumerate(shift_registers, start=1):
        print(f"\nshift_register_{register_number}: {len(chain)} bits")
        for bit, stage in enumerate(chain):
            mux_position = stage.mux.transform.disp
            dff_position = stage.dff.transform.disp
            print(
                f"  bit {bit}: "
                f"MUX {stage.mux.transform} "
                f"({mux_position.x * layout.dbu:.3f}, "
                f"{mux_position.y * layout.dbu:.3f}) um; "
                f"DFF {stage.dff.transform} "
                f"({dff_position.x * layout.dbu:.3f}, "
                f"{dff_position.y * layout.dbu:.3f}) um"
            )

'''
if __name__ == "__main__":
    main()

[RecognizedCell(instance_id=0, cell_type='sky130_fd_sc_hd__tapvpwrvgnd_1', transform=r0 100280,146880, pin_nets={}), RecognizedCell(instance_id=1, cell_type='sky130_fd_sc_hd__tapvpwrvgnd_1', transform=r0 100280,217600, pin_nets={}), RecognizedCell(instance_id=2, cell_type='sky130_fd_sc_hd__tapvpwrvgnd_1', transform=r0 100280,277440, pin_nets={}), RecognizedCell(instance_id=3, cell_type='sky130_fd_sc_hd__tapvpwrvgnd_1', transform=r0 100280,272000, pin_nets={}), RecognizedCell(instance_id=4, cell_type='sky130_fd_sc_hd__tapvpwrvgnd_1', transform=r0 100280,261120, pin_nets={}), RecognizedCell(instance_id=5, cell_type='sky130_fd_sc_hd__tapvpwrvgnd_1', transform=r0 100280,282880, pin_nets={}), RecognizedCell(instance_id=6, cell_type='sky130_fd_sc_hd__tapvpwrvgnd_1', transform=r0 100280,255680, pin_nets={}), RecognizedCell(instance_id=7, cell_type='sky130_fd_sc_hd__tapvpwrvgnd_1', transform=m0 100280,288320, pin_nets={}), RecognizedCell(instance_id=8, cell_type='sky130_fd_sc_hd__tapvpwrvgnd_1

In [19]:
layout = kdb.Layout()
layout.read(str(GDS_PATH))
top = layout.top_cell()

known_pdk_cells = load_pdk_cell_names()
l2n = extract_conductor_nets(layout, top)

l2n

In [21]:
"""Recover schematic-only elements, pins, and shift registers from GDS.

The ``elements`` array contains only functional cells declared by the installed
SKY130 Liberty library. Physical-only fillers, taps, decaps, end caps, and raw
transistor geometry are excluded.
"""

from __future__ import annotations

import argparse
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import klayout.db as kdb

try:
    from .klayoutexmp import (
        DFFR,
        MUX2,
        LogicDefinition,
        RecognizedCell,
        extract_conductor_nets,
        load_liberty_logic,
        load_pdk_cell_names,
        recognize_schematic_cells,
        single_pin_net,
    )
except ImportError:
    from klayoutexmp import (
        DFFR,
        MUX2,
        LogicDefinition,
        RecognizedCell,
        extract_conductor_nets,
        load_liberty_logic,
        load_pdk_cell_names,
        recognize_schematic_cells,
        single_pin_net,
    )


DEFAULT_GDS = Path("../puzzle.gds")
LABEL_DATATYPE = 5
LABEL_CONDUCTORS = {
    67: "li",
    68: "met1",
    69: "met2",
    70: "met3",
    71: "met4",
    72: "met5",
}
POWER_PIN_NAMES = {"VGND", "VPWR", "VNB", "VPB"}


@dataclass
class SchematicElement:
    """One placed standard-cell instance needed by a logical schematic."""

    instance_id: int
    module: str
    symbol_kind: str
    transform: kdb.Trans
    bbox: kdb.Box
    input_pins: dict[str, int]
    output_pins: dict[str, int]
    output_functions: dict[str, str]

    def pin_net(self, pin: str) -> int | None:
        if pin in self.input_pins:
            return self.input_pins[pin]
        return self.output_pins.get(pin)


@dataclass(frozen=True)
class PinReference:
    instance_id: int
    module: str
    pin: str


@dataclass(frozen=True)
class TopInputPin:
    name: str
    net: int
    conductor: str
    point: kdb.Point
    loads: tuple[PinReference, ...]


@dataclass
class RegisterStage:
    mux: SchematicElement
    dff: SchematicElement


@dataclass
class ShiftRegister:
    stages: list[RegisterStage]
    serial_input_net: int
    selector_net: int
    reset_net: int | None
    clock_nets: set[int]

    @property
    def width(self) -> int:
        return len(self.stages)

    @property
    def q_nets(self) -> list[int]:
        return [stage.dff.output_pins["Q"] for stage in self.stages]


@dataclass(frozen=True)
class ShiftRegisterInputPins:
    serial_input: int
    selector: int
    reset: int | None
    clocks: tuple[int, ...]


@dataclass(frozen=True)
class ShiftRegisterOutput:
    bit: int
    net: int
    source: PinReference
    destinations: tuple[PinReference, ...]


@dataclass
class RecoveryContext:
    layout: kdb.Layout
    top: kdb.Cell
    l2n: kdb.LayoutToNetlist
    logic: dict[str, LogicDefinition]
    elements: list[SchematicElement]


def _resolved_pin_net(cell: RecognizedCell, pin: str) -> int | None:
    """Resolve exactly one extracted net for a physical standard-cell pin."""
    return single_pin_net(cell, pin)


def build_schematic_element_array(
    recognized: list[RecognizedCell],
    logic: dict[str, LogicDefinition],
) -> list[SchematicElement]:
    """Convert recognized PDK instances into directed schematic elements."""
    elements: list[SchematicElement] = []

    for cell in recognized:
        definition = logic.get(cell.cell_type)
        if definition is None:
            continue

        input_pins = {
            pin: net
            for pin in definition.input_pins
            if pin not in POWER_PIN_NAMES
            if (net := _resolved_pin_net(cell, pin)) is not None
        }
        output_pins = {
            pin: net
            for pin in definition.output_functions
            if pin not in POWER_PIN_NAMES
            if (net := _resolved_pin_net(cell, pin)) is not None
        }

        # A Liberty function is the criterion for being useful on the logical
        # schematic. It automatically excludes filler/tap/decap masters.
        if not output_pins:
            continue

        elements.append(
            SchematicElement(
                instance_id=cell.instance_id,
                module=cell.cell_type,
                symbol_kind=cell.symbol_kind,
                transform=cell.transform,
                bbox=cell.bbox,
                input_pins=input_pins,
                output_pins=output_pins,
                output_functions=dict(definition.output_functions),
            )
        )

    return elements


def recover_schematic(gds_path: Path = DEFAULT_GDS) -> RecoveryContext:
    """Read GDS and construct the schematic-only functional element array."""
    if not gds_path.is_file():
        raise FileNotFoundError(f"GDS not found: {gds_path}")

    layout = kdb.Layout()
    layout.read(str(gds_path))
    top = layout.top_cell()
    logic = load_liberty_logic(load_pdk_cell_names())
    l2n = extract_conductor_nets(layout, top)
    recognized = recognize_schematic_cells(layout, top, l2n, logic)
    elements = build_schematic_element_array(recognized, logic)
    return RecoveryContext(layout, top, l2n, logic, elements)


def build_load_index(
    elements: list[SchematicElement],
) -> dict[int, list[PinReference]]:
    loads: dict[int, list[PinReference]] = {}
    for element in elements:
        for pin, net in element.input_pins.items():
            loads.setdefault(net, []).append(
                PinReference(element.instance_id, element.module, pin)
            )
    return loads


def build_driver_index(
    elements: list[SchematicElement],
) -> dict[int, list[PinReference]]:
    drivers: dict[int, list[PinReference]] = {}
    for element in elements:
        for pin, net in element.output_pins.items():
            drivers.setdefault(net, []).append(
                PinReference(element.instance_id, element.module, pin)
            )
    return drivers


def find_input_pins(context: RecoveryContext) -> list[TopInputPin]:
    """Find named top-level inputs: labeled nets with loads and no cell driver."""
    loads = build_load_index(context.elements)
    drivers = build_driver_index(context.elements)
    found: dict[tuple[str, int], TopInputPin] = {}

    for gds_layer, conductor in LABEL_CONDUCTORS.items():
        label_layer = context.layout.layer(gds_layer, LABEL_DATATYPE)
        conductor_region = context.l2n.polygons_by_name(conductor)

        for shape in context.top.shapes(label_layer).each():
            if not shape.is_text():
                continue

            text = shape.text
            name = text.string
            if name in POWER_PIN_NAMES:
                continue

            point = text.trans.disp
            extracted_net = context.l2n.probe_net(conductor_region, point)
            if extracted_net is None:
                continue

            net = extracted_net.cluster_id
            if net not in loads or net in drivers:
                continue

            found[(name, net)] = TopInputPin(
                name=name,
                net=net,
                conductor=conductor,
                point=point,
                loads=tuple(loads[net]),
            )

    return sorted(found.values(), key=lambda pin: pin.name)


def _is_register_stage(
    mux: SchematicElement,
    dff: SchematicElement,
) -> bool:
    """Recognize hold/shift MUX feedback connected to one D flip-flop."""
    return (
        mux.module == MUX2
        and dff.module == DFFR
        and mux.output_pins.get("X") == dff.input_pins.get("D")
        and mux.input_pins.get("A0") == dff.output_pins.get("Q")
    )


def find_shift_registers(
    elements: list[SchematicElement],
) -> list[ShiftRegister]:
    """Find ordered MUX+DFF chains, allowing locally buffered clock nets."""
    muxes = [element for element in elements if element.module == MUX2]
    dffs = [element for element in elements if element.module == DFFR]
    stages = [
        RegisterStage(mux, dff)
        for mux in muxes
        for dff in dffs
        if _is_register_stage(mux, dff)
    ]

    successors: dict[int, list[int]] = {
        index: [] for index in range(len(stages))
    }
    predecessors: dict[int, list[int]] = {
        index: [] for index in range(len(stages))
    }

    for previous_index, previous in enumerate(stages):
        previous_q = previous.dff.output_pins.get("Q")
        for following_index, following in enumerate(stages):
            if previous_index == following_index:
                continue
            if previous_q == following.mux.input_pins.get("A1"):
                successors[previous_index].append(following_index)
                predecessors[following_index].append(previous_index)

    registers: list[ShiftRegister] = []
    for root in (index for index in predecessors if not predecessors[index]):
        indexes: list[int] = []
        current = root
        while current not in indexes:
            indexes.append(current)
            if len(successors[current]) != 1:
                break
            current = successors[current][0]

        chain = [stages[index] for index in indexes]
        if len(chain) < 2:
            continue

        selector_nets = {
            stage.mux.input_pins.get("S") for stage in chain
        }
        reset_nets = {
            stage.dff.input_pins.get("RESET_B") for stage in chain
        }
        if len(selector_nets) != 1 or len(reset_nets) != 1:
            continue

        selector = next(iter(selector_nets))
        serial_input = chain[0].mux.input_pins.get("A1")
        if selector is None or serial_input is None:
            continue

        registers.append(
            ShiftRegister(
                stages=chain,
                serial_input_net=serial_input,
                selector_net=selector,
                reset_net=next(iter(reset_nets)),
                clock_nets={
                    net
                    for stage in chain
                    if (net := stage.dff.input_pins.get("CLK")) is not None
                },
            )
        )

    return sorted(registers, key=lambda register: -register.width)


def find_shift_register_input_pins(
    register: ShiftRegister,
) -> ShiftRegisterInputPins:
    """Return the virtual input pins of one recovered shift-register block."""
    return ShiftRegisterInputPins(
        serial_input=register.serial_input_net,
        selector=register.selector_net,
        reset=register.reset_net,
        clocks=tuple(sorted(register.clock_nets)),
    )


def find_shift_register_outputs(
    register: ShiftRegister,
    elements: list[SchematicElement],
) -> list[ShiftRegisterOutput]:
    """Return Q nets consumed by schematic elements outside the register."""
    internal_ids = {
        element.instance_id
        for stage in register.stages
        for element in (stage.mux, stage.dff)
    }
    load_index = build_load_index(elements)
    outputs: list[ShiftRegisterOutput] = []

    for bit, stage in enumerate(register.stages):
        q_net = stage.dff.output_pins["Q"]
        destinations = tuple(
            endpoint
            for endpoint in load_index.get(q_net, [])
            if endpoint.instance_id not in internal_ids
        )
        if not destinations:
            continue

        outputs.append(
            ShiftRegisterOutput(
                bit=bit,
                net=q_net,
                source=PinReference(
                    stage.dff.instance_id,
                    stage.dff.module,
                    "Q",
                ),
                destinations=destinations,
            )
        )

    return outputs


def short_module(module: str) -> str:
    return module.removeprefix("sky130_fd_sc_hd__")


In [22]:
context = recover_schematic()
context = recover_schematic()
inputs = find_input_pins(context)
registers = find_shift_registers(context.elements)
for kind, count in sorted(
    Counter(element.symbol_kind for element in context.elements).items()
):
    print(f"  {kind}: {count}")

print("\nTop-level input pins:")
for pin in inputs:
    print(f"  {pin.name}: net_{pin.net}; loads={len(pin.loads)}")

print(f"\nShift registers: {len(registers)}")
for register_index, register in enumerate(registers, start=1):
    block_inputs = find_shift_register_input_pins(register)
    outputs = find_shift_register_outputs(register, context.elements)
    print(f"  shift_register_{register_index}: {register.width} bits")
    print(f"    serial input: net_{block_inputs.serial_input}")
    print(f"    selector: net_{block_inputs.selector}")
    print(f"    reset: net_{block_inputs.reset}")
    print(
        "    clocks: "
        + ", ".join(f"net_{net}" for net in block_inputs.clocks)
    )
    print(
        "    Q order: "
        + " -> ".join(f"net_{net}" for net in register.q_nets)
    )
    print("    external outputs:")
    for output in outputs:
        destinations = ", ".join(
            f"u_{pin.instance_id}.{pin.pin}({short_module(pin.module)})"
            for pin in output.destinations
        )
        print(
            f"      Q[{output.bit}] / net_{output.net} -> {destinations}"
        )

  AND gate: 103
  NAND gate: 81
  NOR gate: 62
  OR gate: 52
  XNOR gate: 29
  XOR gate: 21
  buffer: 33
  compound logic gate: 203
  constant source: 6
  flip-flop: 92
  inverter: 25
  multiplexer: 21

Top-level input pins:
  I: net_520; loads=45
  clk: net_1608; loads=1
  enable: net_1186; loads=1
  rst_n: net_322; loads=88

Shift registers: 1
  shift_register_1: 12 bits
    serial input: net_520
    selector: net_604
    reset: net_322
    clocks: net_1070, net_1313, net_1348, net_1575
    Q order: net_1577 -> net_1547 -> net_1596 -> net_1458 -> net_1402 -> net_1349 -> net_1309 -> net_1212 -> net_1248 -> net_1308 -> net_1358 -> net_1512
    external outputs:
      Q[0] / net_1577 -> u_4999.A2(a22o_2)
      Q[9] / net_1308 -> u_5229.A2(a221o_2)
      Q[10] / net_1358 -> u_5229.B2(a221o_2)
      Q[11] / net_1512 -> u_4999.B2(a22o_2)


In [23]:
context.elements

[SchematicElement(instance_id=51, module='sky130_fd_sc_hd__clkbuf_4', symbol_kind='buffer', transform=r0 97520,70720, bbox=(97330,70480;100470,73680), input_pins={'A': 348}, output_pins={'X': 683}, output_functions={'X': '(A)'}),
 SchematicElement(instance_id=52, module='sky130_fd_sc_hd__clkbuf_8', symbol_kind='buffer', transform=r180 102580,76160, bbox=(97330,73200;102770,76400), input_pins={'A': 695}, output_pins={'X': 348}, output_functions={'X': '(A)'}),
 SchematicElement(instance_id=63, module='sky130_fd_sc_hd__a211o_2', symbol_kind='compound logic gate', transform=m0 174800,119680, bbox=(174610,116720;178670,119920), input_pins={'A1': 749, 'A2': 1136, 'B1': 1041, 'C1': 1083}, output_pins={'X': 1135}, output_functions={'X': '(A1&A2) | (B1) | (C1)'}),
 SchematicElement(instance_id=67, module='sky130_fd_sc_hd__a211oi_2', symbol_kind='compound logic gate', transform=m0 170200,119680, bbox=(170010,116720;174990,119920), input_pins={'A1': 762, 'A2': 749, 'B1': 751, 'C1': 848}, output_p

In [9]:
for elem in context.elements:
    elem_inputs = elem.input_pins.values()
    if(1577 in elem_inputs or 1308 in elem_inputs or 1358 in elem_inputs or 1512 in elem_inputs):
        print(elem)

SchematicElement(instance_id=4999, module='sky130_fd_sc_hd__a22o_2', symbol_kind='compound logic gate', transform=r180 86940,163200, bbox=(83070,160240;87130,163440), input_pins={'A1': 1029, 'A2': 1577, 'B1': 978, 'B2': 1512}, output_pins={'X': 1456}, output_functions={'X': '(B1&B2) | (A1&A2)'})
SchematicElement(instance_id=5008, module='sky130_fd_sc_hd__mux2_1', symbol_kind='multiplexer', transform=r0 77740,157760, bbox=(77550,157520;82070,160720), input_pins={'A0': 1547, 'A1': 1577, 'S': 604}, output_pins={'X': 1544}, output_functions={'X': '(A0&!S) | (A1&S)'})
SchematicElement(instance_id=5229, module='sky130_fd_sc_hd__a221o_2', symbol_kind='compound logic gate', transform=r180 72680,152320, bbox=(68350,149360;72870,152560), input_pins={'A1': 1002, 'A2': 1308, 'B1': 926, 'B2': 1358, 'C1': 1456}, output_pins={'X': 1454}, output_functions={'X': '(B1&B2) | (A1&A2) | (C1)'})
SchematicElement(instance_id=5340, module='sky130_fd_sc_hd__mux2_1', symbol_kind='multiplexer', transform=r180 92

In [14]:
for elem in context.elements:
    elem_outputs = elem.output_pins.values()
    if(169 in elem_outputs or 114 in elem_outputs or 1001 in elem_outputs or 547 in elem_outputs):
        print(elem)

SchematicElement(instance_id=4659, module='sky130_fd_sc_hd__dfrtp_2', symbol_kind='flip-flop', transform=r180 35880,146880, bbox=(26030,143920;36070,147120), input_pins={'CLK': 1348, 'D': 1307, 'RESET_B': 322}, output_pins={'Q': 1001}, output_functions={'Q': 'IQ'})
SchematicElement(instance_id=4661, module='sky130_fd_sc_hd__dfrtp_2', symbol_kind='flip-flop', transform=r0 28980,157760, bbox=(28790,157520;38830,160720), input_pins={'CLK': 1348, 'D': 1568, 'RESET_B': 322}, output_pins={'Q': 547}, output_functions={'Q': 'IQ'})
SchematicElement(instance_id=4662, module='sky130_fd_sc_hd__dfrtp_2', symbol_kind='flip-flop', transform=m0 26220,152320, bbox=(26030,149360;36070,152560), input_pins={'CLK': 1348, 'D': 1448, 'RESET_B': 322}, output_pins={'Q': 169}, output_functions={'Q': 'IQ'})
SchematicElement(instance_id=4838, module='sky130_fd_sc_hd__dfrtp_2', symbol_kind='flip-flop', transform=r0 30820,146880, bbox=(30630,146640;40670,149840), input_pins={'CLK': 1348, 'D': 1394, 'RESET_B': 322},

In [23]:
for elem in context.elements:
    elem_outputs = elem.output_pins.values()
    if(1001 in elem_outputs or 547 in elem_outputs or 114 in elem_outputs):
        print(elem)

SchematicElement(instance_id=4659, module='sky130_fd_sc_hd__dfrtp_2', symbol_kind='flip-flop', transform=r180 35880,146880, bbox=(26030,143920;36070,147120), input_pins={'CLK': 1348, 'D': 1307, 'RESET_B': 322}, output_pins={'Q': 1001}, output_functions={'Q': 'IQ'})
SchematicElement(instance_id=4661, module='sky130_fd_sc_hd__dfrtp_2', symbol_kind='flip-flop', transform=r0 28980,157760, bbox=(28790,157520;38830,160720), input_pins={'CLK': 1348, 'D': 1568, 'RESET_B': 322}, output_pins={'Q': 547}, output_functions={'Q': 'IQ'})
SchematicElement(instance_id=4838, module='sky130_fd_sc_hd__dfrtp_2', symbol_kind='flip-flop', transform=r0 30820,146880, bbox=(30630,146640;40670,149840), input_pins={'CLK': 1348, 'D': 1394, 'RESET_B': 322}, output_pins={'Q': 114}, output_functions={'Q': 'IQ'})


In [24]:
LABEL_DATATYPE = 5

LABEL_CONDUCTORS = {
    67: "li",
    68: "met1",
    69: "met2",
    70: "met3",
    71: "met4",
    72: "met5",
}

def get_success_net(context) -> int:
    for gds_layer, conductor in LABEL_CONDUCTORS.items():
        label_layer = context.layout.layer(
            gds_layer,
            LABEL_DATATYPE,
        )
        conductor_region = context.l2n.polygons_by_name(conductor)

        for shape in context.top.shapes(label_layer).each():
            if not shape.is_text():
                continue

            if shape.text.string != "success":
                continue

            extracted_net = context.l2n.probe_net(
                conductor_region,
                shape.text.trans.disp,
            )

            if extracted_net is not None:
                return extracted_net.cluster_id

    raise ValueError("GDS label 'success' was not found")

def find_success_cell(elements):
    # Puzzle-specific value previously recovered from the GDS label "success".
    SUCCESS_NET = 2380

    matches = [
        element
        for element in elements
        if SUCCESS_NET in element.output_pins.values()
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one cell driving net_{SUCCESS_NET}, "
            f"found {len(matches)}"
        )

    return matches[0]


success_cell = find_success_cell(context.elements)
print(success_cell)
success_net = get_success_net(context)
print(success_net)



SchematicElement(instance_id=228, module='sky130_fd_sc_hd__dfrtp_2', symbol_kind='flip-flop', transform=m0 172040,282880, bbox=(171850,279920;181890,283120), input_pins={'CLK': 2212, 'D': 2543, 'RESET_B': 322}, output_pins={'Q': 2380}, output_functions={'Q': 'IQ'})
2380


In [36]:
def find_element_by_output_net(
    elements: list[SchematicElement],
    output_net: int,
) -> SchematicElement:
    """Return the single element that drives the specified net."""

    matches = [
        element
        for element in elements
        if output_net in element.output_pins.values()
    ]

    if not matches:
        raise ValueError(f"No element drives net_{output_net}")

    if len(matches) > 1:
        raise ValueError(
            f"Multiple elements drive net_{output_net}: "
            f"{[element.instance_id for element in matches]}"
        )

    return matches[0]

print(find_element_by_output_net(context.elements, 2543))
print(find_element_by_output_net(context.elements, 828))

SchematicElement(instance_id=233, module='sky130_fd_sc_hd__a32o_2', symbol_kind='compound logic gate', transform=m90 177100,277440, bbox=(172770,277200;177290,280400), input_pins={'A1': 1361, 'A2': 2487, 'A3': 2542, 'B1': 2380, 'B2': 2520}, output_pins={'X': 2543}, output_functions={'X': '(A1&A2&A3) | (B1&B2)'})
SchematicElement(instance_id=5638, module='sky130_fd_sc_hd__dfrtp_2', symbol_kind='flip-flop', transform=r0 80960,92480, bbox=(80770,92240;90810,95440), input_pins={'CLK': 829, 'D': 831, 'RESET_B': 322}, output_pins={'Q': 828}, output_functions={'Q': 'IQ'})


In [57]:
from collections import defaultdict, deque
from dataclasses import dataclass


@dataclass
class BacktraceResult:
    combinational_levels: dict[int, list[SchematicElement]]
    flip_flop_levels: dict[int, list[SchematicElement]]
    undriven_nets: dict[int, list[int]]
    visited_nets: set[int]


def backtrace_to_flip_flops(
    elements: list[SchematicElement],
    start_net: int,
) -> BacktraceResult:
    """Trace a net backward through combinational cells until FF boundaries."""

    # output net -> elements driving that net
    drivers_by_net: dict[int, list[SchematicElement]] = defaultdict(list)

    for element in elements:
        for output_net in element.output_pins.values():
            drivers_by_net[output_net].append(element)

    queue = deque([(start_net, 0)])
    visited_nets: set[int] = set()
    visited_combinational_cells: set[int] = set()
    visited_flip_flops: set[int] = set()

    combinational_levels: dict[int, list[SchematicElement]] = defaultdict(list)
    flip_flop_levels: dict[int, list[SchematicElement]] = defaultdict(list)
    undriven_nets: dict[int, list[int]] = defaultdict(list)

    while queue:
        current_net, level = queue.popleft()

        if current_net in visited_nets:
            continue

        visited_nets.add(current_net)
        drivers = drivers_by_net.get(current_net, [])

        # No element output drives this net. It may be a top-level input,
        # constant/floating conductor or another extraction boundary.
        if not drivers:
            undriven_nets[level].append(current_net)
            continue

        for element in drivers:
            # Sequential boundary: record the FF but do not continue through
            # its D, CLK, reset or set pins.
            if element.symbol_kind in {"flip-flop", "latch"}:
                if element.instance_id not in visited_flip_flops:
                    visited_flip_flops.add(element.instance_id)
                    flip_flop_levels[level].append(element)
                continue

            # Constant sources also terminate a branch because they have no
            # meaningful upstream input logic.
            if element.symbol_kind == "constant source":
                continue

            if element.instance_id in visited_combinational_cells:
                continue

            visited_combinational_cells.add(element.instance_id)
            combinational_levels[level].append(element)

            # Move one level backward through every combinational input.
            for input_net in element.input_pins.values():
                queue.append((input_net, level + 1))

    return BacktraceResult(
        combinational_levels={
            level: sorted(cells, key=lambda cell: cell.instance_id)
            for level, cells in sorted(combinational_levels.items())
        },
        flip_flop_levels={
            level: sorted(cells, key=lambda cell: cell.instance_id)
            for level, cells in sorted(flip_flop_levels.items())
        },
        undriven_nets={
            level: sorted(set(nets))
            for level, nets in sorted(undriven_nets.items())
        },
        visited_nets=visited_nets,
    )

In [62]:
import json
from pathlib import Path


def find_report_path():
    """Works whether the notebook runs from the project root or scripts/."""
    possible_paths = [
        Path("scripts/recovered_register_groups.json"),
        Path("recovered_register_groups.json"),
    ]

    for path in possible_paths:
        if path.exists():
            return path

    raise FileNotFoundError("Cannot find recovered_register_groups.json")


def build_register_name_map(elements):
    """
    Returns:
        {
            instance_id: "register_XX[bit]",
            ...
        }
    """

    with find_report_path().open() as file:
        report = json.load(file)

    register_names = {}

    # Known multi-bit register groups.
    for group_number, group in enumerate(
        report["register_groups"],
        start=1,
    ):
        # Shift register has a proven bit_order.
        # Other groups use the stored flip_flops order.
        bit_order = group.get("bit_order") or group["flip_flops"]

        for bit, instance_id in enumerate(bit_order):
            register_names[instance_id] = (
                f"register_{group_number:02d}[{bit}]"
            )

    # Treat every ungrouped FF as a one-bit register.
    next_group_number = len(report["register_groups"]) + 1

    flip_flops = sorted(
        (
            element
            for element in elements
            if element.symbol_kind in {"flip-flop", "latch"}
        ),
        key=lambda element: element.instance_id,
    )

    for element in flip_flops:
        if element.instance_id not in register_names:
            register_names[element.instance_id] = (
                f"register_{next_group_number:02d}[0]"
            )
            next_group_number += 1

    return register_names

In [76]:
trace = backtrace_to_flip_flops(
    context.elements,
    start_net=1360,  # u_228.D, the logic generating success
)

for level in sorted(
    set(trace.combinational_levels) | set(trace.flip_flop_levels)
):
    print(f"\nLevel {level}")

    for element in trace.combinational_levels.get(level, []):
        print(
            f"  combinational: u_{element.output_functions} "
            f" {element.input_pins} "
            f"{element.output_pins}"
        )

    for element in trace.flip_flop_levels.get(level, []):
        print(
            f"  FF boundary: u_{element.instance_id} "
            f"{element.module}; Q={element.output_pins.get('Q')}"
        )


Level 0
  FF boundary: u_5337 sky130_fd_sc_hd__dfrtp_2; Q=1360


In [79]:
print(find_element_by_output_net(context.elements, 1360))

SchematicElement(instance_id=5337, module='sky130_fd_sc_hd__dfrtp_2', symbol_kind='flip-flop', transform=m0 87860,152320, bbox=(87670,149360;97710,152560), input_pins={'CLK': 1313, 'D': 1466, 'RESET_B': 322}, output_pins={'Q': 1360}, output_functions={'Q': 'IQ'})


In [80]:
trace = backtrace_to_flip_flops(
    context.elements,
    start_net=1466,  # u_228.D, the logic generating success
)

for level in sorted(
    set(trace.combinational_levels) | set(trace.flip_flop_levels)
):
    print(f"\nLevel {level}")

    for element in trace.combinational_levels.get(level, []):
        print(
            f"  combinational: u_{element.output_functions} "
            f" {element.input_pins} "
            f"{element.output_pins}"
        )

    for element in trace.flip_flop_levels.get(level, []):
        print(
            f"  FF boundary: u_{element.instance_id} "
            f"{element.module}; Q={element.output_pins.get('Q')}"
        )


Level 0
  combinational: u_{'X': '(A1&A2&A3) | (B1)'}  {'A1': 520, 'A2': 604, 'A3': 1454, 'B1': 1360} {'X': 1466}

Level 1
  combinational: u_{'X': '(!A_N&B)'}  {'A_N': 1968, 'B': 1186} {'X': 604}
  combinational: u_{'X': '(B1&B2) | (A1&A2) | (C1)'}  {'A1': 1002, 'A2': 1308, 'B1': 926, 'B2': 1358, 'C1': 1456} {'X': 1454}
  FF boundary: u_5337 sky130_fd_sc_hd__dfrtp_2; Q=1360

Level 2
  combinational: u_{'X': '(B1&B2) | (A1&A2)'}  {'A1': 1029, 'A2': 1577, 'B1': 978, 'B2': 1512} {'X': 1456}
  combinational: u_{'X': '(A) | (B) | (!C_N) | (!D_N)'}  {'A': 169, 'B': 1001, 'C_N': 547, 'D_N': 114} {'X': 1002}
  FF boundary: u_4182 sky130_fd_sc_hd__dfrtp_2; Q=1968
  FF boundary: u_5339 sky130_fd_sc_hd__dfrtp_2; Q=1358
  FF boundary: u_5344 sky130_fd_sc_hd__dfrtp_2; Q=1308

Level 3
  combinational: u_{'X': '(A) | (B) | (C) | (D)'}  {'A': 169, 'B': 114, 'C': 1001, 'D': 547} {'X': 1029}
  combinational: u_{'X': '(A)'}  {'A': 1029} {'X': 978}
  FF boundary: u_4659 sky130_fd_sc_hd__dfrtp_2; Q=1001


In [82]:
trace = backtrace_to_flip_flops(
    context.elements,
    start_net=2543,  # u_228.D: logic generating success
)

register_names = build_register_name_map(context.elements)

all_levels = (
    set(trace.combinational_levels)
    | set(trace.flip_flop_levels)
)

for level in sorted(all_levels):
    print(f"\nLevel {level}")

    for element in trace.combinational_levels.get(level, []):
        print(
            f"  combinational: "
            f"u_{element.instance_id} "
            f"{element.module}; "
            f"inputs={element.input_pins}; "
            f"outputs={element.output_pins}; "
            f"function={element.output_functions}"
        )

    for element in trace.flip_flop_levels.get(level, []):
        register_name = register_names[element.instance_id]
        q_net = element.output_pins.get("Q")

        print(
            f"  {register_name}: "
            f"Q=net_{q_net} "
            f"(physical cell u_{element.instance_id})"
        )


Level 0
  combinational: u_233 sky130_fd_sc_hd__a32o_2; inputs={'A1': 1361, 'A2': 2487, 'A3': 2542, 'B1': 2380, 'B2': 2520}; outputs={'X': 2543}; function={'X': '(A1&A2&A3) | (B1&B2)'}

Level 1
  combinational: u_406 sky130_fd_sc_hd__nand2b_2; inputs={'A_N': 2329, 'B': 1968}; outputs={'Y': 2520}; function={'Y': '(A_N) | (!B)'}
  combinational: u_450 sky130_fd_sc_hd__and4b_2; inputs={'A_N': 2329, 'B': 1968, 'C': 164, 'D': 1573}; outputs={'X': 2542}; function={'X': '(!A_N&B&C&D)'}
  combinational: u_460 sky130_fd_sc_hd__and2_2; inputs={'A': 855, 'B': 367}; outputs={'X': 2487}; function={'X': '(A&B)'}
  combinational: u_5532 sky130_fd_sc_hd__inv_2; inputs={'A': 1360}; outputs={'Y': 1361}; function={'Y': '(!A)'}
  register_09[2]: Q=net_2380 (physical cell u_228)

Level 2
  combinational: u_5013 sky130_fd_sc_hd__and3_2; inputs={'A': 1506, 'B': 1539, 'C': 1500}; outputs={'X': 1573}; function={'X': '(A&B&C)'}
  combinational: u_5836 sky130_fd_sc_hd__inv_2; inputs={'A': 828}; outputs={'Y': 85

In [155]:
group_7 = dict()

for item in register_names.items():
    if item[1][0:11] == 'register_10':
        group_7[item[0]] = item[1]

group_7_cells = dict()

for elem in context.elements:
    ids = [r for r in group_7.keys()]
    if elem.instance_id in ids:
        group_7_cells[group_7[elem.instance_id]] = elem

for item in group_7_cells.items():
    print(item[0], item[1].input_pins['D'])

register_10[0] 1132
register_10[1] 840
register_10[2] 1039
register_10[3] 892
register_10[4] 1109
register_10[5] 857
register_10[6] 939
register_10[7] 1015


In [172]:
trace = backtrace_to_flip_flops(
    context.elements,
    start_net=1131,  # u_228.D: logic generating success
)

register_names = build_register_name_map(context.elements)

all_levels = (
    set(trace.combinational_levels)
    | set(trace.flip_flop_levels)
)

for level in sorted(all_levels):
    print(f"\nLevel {level}")

    for element in trace.combinational_levels.get(level, []):
        print(
            f"  combinational: "
            f"u_{element.instance_id} "
            f"{element.module}; "
            f"inputs={element.input_pins}; "
            f"outputs={element.output_pins}; "
            f"function={element.output_functions}"
        )

    for element in trace.flip_flop_levels.get(level, []):
        register_name = register_names[element.instance_id]
        q_net = element.output_pins.get("Q")

        print(
            f"  {register_name}: "
            f"Q=net_{q_net} "
            f"(physical cell u_{element.instance_id})"
        )

#register_10[0] 1132
#register_10[1] 840
#register_10[2] 1039
#register_10[3] 892
#register_10[4] 1109
#register_10[5] 857
#register_10[6] 939
#register_10[7] 1015


Level 0
  combinational: u_2707 sky130_fd_sc_hd__and4b_2; inputs={'A_N': 584, 'B': 559, 'C': 643, 'D': 497}; outputs={'X': 1131}; function={'X': '(!A_N&B&C&D)'}

Level 1
  combinational: u_6978 sky130_fd_sc_hd__o211a_2; inputs={'A1': 400, 'A2': 466, 'B1': 509, 'C1': 417}; outputs={'X': 584}; function={'X': '(A1&B1&C1) | (A2&B1&C1)'}
  combinational: u_6987 sky130_fd_sc_hd__o211a_2; inputs={'A1': 400, 'A2': 549, 'B1': 465, 'C1': 417}; outputs={'X': 559}; function={'X': '(A1&B1&C1) | (A2&B1&C1)'}
  combinational: u_7563 sky130_fd_sc_hd__o211a_2; inputs={'A1': 400, 'A2': 419, 'B1': 355, 'C1': 417}; outputs={'X': 497}; function={'X': '(A1&B1&C1) | (A2&B1&C1)'}
  combinational: u_7824 sky130_fd_sc_hd__o211a_2; inputs={'A1': 264, 'A2': 404, 'B1': 417, 'C1': 403}; outputs={'X': 643}; function={'X': '(A1&B1&C1) | (A2&B1&C1)'}

Level 2
  combinational: u_6948 sky130_fd_sc_hd__o311a_2; inputs={'A1': 202, 'A2': 12, 'A3': 45, 'B1': 74, 'C1': 216}; outputs={'X': 264}; function={'X': '(A1&B1&C1) | 

In [122]:
import json
import re
from pathlib import Path


SEQUENTIAL_KINDS = {"flip-flop", "latch"}
IDENTIFIER_RE = re.compile(r"[A-Za-z_]\w*")


def find_register_report():
    for path in (
        Path("scripts/recovered_register_groups.json"),
        Path("recovered_register_groups.json"),
    ):
        if path.exists():
            return path

    raise FileNotFoundError("Cannot find recovered_register_groups.json")


def build_register_output_names(elements):
    """
    Map electrical Q nets to names such as:

        net_2329 -> register_09[0]
        net_1968 -> register_16[0]
        net_1360 -> register_17[0]
    """

    with find_register_report().open() as file:
        report = json.load(file)

    elements_by_id = {
        element.instance_id: element
        for element in elements
    }

    register_name_by_instance = {}

    # Recovered multi-bit groups.
    for group_number, group in enumerate(
        report["register_groups"],
        start=1,
    ):
        bit_order = group.get("bit_order") or group["flip_flops"]

        for bit, instance_id in enumerate(bit_order):
            register_name_by_instance[instance_id] = (
                f"register_{group_number:02d}[{bit}]"
            )

    # FFs not assigned to a multi-bit group become one-bit registers.
    next_group = len(report["register_groups"]) + 1

    sequential_elements = sorted(
        (
            element
            for element in elements
            if element.symbol_kind in SEQUENTIAL_KINDS
        ),
        key=lambda element: element.instance_id,
    )

    for element in sequential_elements:
        if element.instance_id not in register_name_by_instance:
            register_name_by_instance[element.instance_id] = (
                f"register_{next_group:02d}[0]"
            )
            next_group += 1

    # Convert instance names to output-net names.
    register_output_by_net = {}

    for instance_id, register_name in register_name_by_instance.items():
        element = elements_by_id[instance_id]

        for output_pin, output_net in element.output_pins.items():
            if output_pin in {"Q", "IQ"}:
                expression = register_name
            elif output_pin in {"Q_N", "IQ_N"}:
                expression = f"!({register_name})"
            else:
                expression = f"{register_name}.{output_pin}"

            register_output_by_net[output_net] = expression

    return register_output_by_net

In [123]:
def equation_from_trace(
    trace,
    elements,
    start_net,
    pure_net_names=None,
    constant_nets=None,
    print_levels=True,
):
    """
    Expand start_net through all combinational cells in TraceResult.

    Expansion stops at:

      1. Register Q outputs
      2. Nets listed in pure_net_names
      3. Nets listed in constant_nets

    Returns a dictionary containing the final equation and equations by level.
    """

    pure_net_names = dict(pure_net_names or {})
    constant_nets = dict(constant_nets or {})

    register_outputs = build_register_output_names(elements)

    # Only use combinational cells contained in this trace.
    combinational_driver = {}

    for level, level_elements in trace.combinational_levels.items():
        for element in level_elements:
            for output_pin, output_net in element.output_pins.items():
                previous = combinational_driver.get(output_net)

                if previous is not None:
                    old_element, _ = previous
                    raise ValueError(
                        f"net_{output_net} has multiple traced drivers: "
                        f"u_{old_element.instance_id} and "
                        f"u_{element.instance_id}"
                    )

                combinational_driver[output_net] = (
                    element,
                    output_pin,
                )

    expanded_cache = {}
    expansion_stack = set()

    def expand_net(net):
        """Recursively expand one net to register/I/O boundaries."""

        # A sequential Q output is always a boundary.
        if net in register_outputs:
            return register_outputs[net]

        # A constant or explicitly named external boundary.
        if net in constant_nets:
            return str(constant_nets[net])

        # If this net has no combinational driver in the trace, it must be
        # an explicitly named input/output boundary.
        driver = combinational_driver.get(net)

        if driver is None:
            if net in pure_net_names:
                return pure_net_names[net]

            raise KeyError(
                f"net_{net} has no driver inside this trace and has no "
                "name. Add it to pure_net_names or constant_nets."
            )

        if net in expanded_cache:
            return expanded_cache[net]

        if net in expansion_stack:
            raise ValueError(
                f"Combinational cycle detected through net_{net}"
            )

        expansion_stack.add(net)

        element, output_pin = driver

        function = element.output_functions.get(output_pin)
        if function is None:
            raise ValueError(
                f"u_{element.instance_id}.{output_pin} has no "
                "Liberty Boolean function"
            )

        # Convert alternate Liberty operators.
        function = function.replace("*", "&").replace("+", "|")

        # Convert simple Liberty postfix inversion: A' -> !A.
        function = re.sub(
            r"\b([A-Za-z_]\w*)'",
            r"!\1",
            function,
        )

        pin_expressions = {
            pin_name: expand_net(input_net)
            for pin_name, input_net in element.input_pins.items()
        }

        function_identifiers = set(
            IDENTIFIER_RE.findall(function)
        )

        unknown_identifiers = (
            function_identifiers - set(pin_expressions)
        )

        if unknown_identifiers:
            raise ValueError(
                f"u_{element.instance_id} {element.module}: "
                f"unknown function identifiers "
                f"{sorted(unknown_identifiers)}"
            )

        def replace_pin(match):
            pin_name = match.group(0)
            return f"({pin_expressions[pin_name]})"

        expression = IDENTIFIER_RE.sub(
            replace_pin,
            function,
        )

        expression = f"({expression})"

        expansion_stack.remove(net)
        expanded_cache[net] = expression

        return expression

    equations_by_level = {}

    all_levels = (
        set(trace.combinational_levels)
        | set(trace.flip_flop_levels)
    )

    for level in sorted(all_levels):
        level_equations = []

        for element in trace.combinational_levels.get(level, []):
            for output_pin, output_net in element.output_pins.items():
                expression = expand_net(output_net)

                equation = (
                    f"net_{output_net} = {expression}"
                )

                level_equations.append(equation)

        for element in trace.flip_flop_levels.get(level, []):
            for output_pin, output_net in element.output_pins.items():
                register_name = register_outputs[output_net]

                level_equations.append(
                    f"net_{output_net} = {register_name}"
                )

        equations_by_level[level] = level_equations

    final_rhs = expand_net(start_net)
    final_lhs = pure_net_names.get(
        start_net,
        f"net_{start_net}",
    )

    result = {
        "lhs": final_lhs,
        "rhs": final_rhs,
        "equation": f"{final_lhs} = {final_rhs}",
        "equations_by_level": equations_by_level,
        "register_outputs": register_outputs,
    }

    if print_levels:
        for level in sorted(equations_by_level):
            print(f"\nLevel {level}")

            for equation in equations_by_level[level]:
                print(f"  {equation}")

        print("\nFinal mathematical equation:")
        print(result["equation"])

    return result

In [170]:
trace = backtrace_to_flip_flops(
    context.elements,
    start_net=2543,
)

pure_net_names = {
    520: "serial_in",
    1186: "enable",
    322: "rst_n",
    1608: "clk",

    # Add recovered output-net IDs when required:
    # output_net_0: "out[0]",
    # output_net_1: "out[1]",
    # 2380: "success",
}

constant_nets = {
    # Example:
    # 123: "1'b1",
    # 456: "1'b0",
}

result = equation_from_trace(
    trace=trace,
    elements=context.elements,
    start_net=2543,
    pure_net_names=pure_net_names,
    constant_nets=constant_nets,
)

#register_10[0] 1132
#register_10[1] 840
#register_10[2] 1039
#register_10[3] 892
#register_10[4] 1109
#register_10[5] 857
#register_10[6] 939
#register_10[7] 1015


Level 0
  net_2543 = (((((!(register_17[0]))))&((((((!(register_06[1]))))&((((register_07[2])&(register_07[1])&(register_07[6])&(((!(register_07[5])&!(register_07[7])&!((((register_07[3])) | ((register_07[0])) | ((register_07[4])))))))))))))&(((!(register_09[0])&(register_16[0])&(((((((((!(register_10[6])&(register_10[3]))))&((((register_10[1])&(((!(register_10[5])))))))&(((!(register_10[0])&(register_10[4]))))&(((!(register_10[7])&(register_10[2])))))))&(((((((register_11[5])&(((!(register_11[6])))))))&(((!(register_11[7])&(register_11[0]))))&(((!(register_11[2])&(register_11[3]))))&((((register_11[4])&(((!(register_11[1]))))))))))&((((((!(register_08[2])&(register_08[0]))))&((((register_08[5])&(((!(register_08[4])))))))&(((!(register_08[3])&(register_08[1]))))))))))&(((((((((!(register_14[0])&(register_14[4]))))&((((register_15[0])&(((!(register_15[4])))))))&(((!(register_14[2])&(register_14[3]))))&(((!(register_14[1])&(register_14[5])))))))&(((((((register_15[7])&(((!(register_15[2

In [26]:
import importlib
from pathlib import Path
import generate_standalone_register_groups as generator

importlib.reload(generator)

verilog, information = generator.generate_full_register_from_backtrace(
    context=context,
    register_group_index=8,
)

Path("generated_register_8.v").write_text(verilog)
print(verilog)

// Stateful RTL recovered from puzzle.gds.
// The first source-register layer is the module boundary.
module recovered_register_08 (
    input wire clk,
    input wire rst_n,
    input wire enable,
    input wire [3:0] register_03,  // source register at backtrace boundary
    input wire [3:0] register_04,  // source register at backtrace boundary
    input wire register_16,  // u_4182.Q at backtrace boundary
    input wire serial_in,
    output reg [5:0] register_08  // recovered register state
);

  // Physical flip-flop to RTL-bit mapping.
  // register_08[0] = u_2299.Q
  // register_08[1] = u_2301.Q
  // register_08[2] = u_2542.Q
  // register_08[3] = u_2574.Q
  // register_08[4] = u_2708.Q
  // register_08[5] = u_2710.Q

  // Recovered combinational nets.
  wire net_1;
  wire net_2;
  wire net_4;
  wire net_5;
  wire net_6;
  wire net_8;
  wire net_9;
  wire net_11;
  wire net_12;
  wire net_14;
  wire net_15;
  wire net_16;
  wire net_17;
  wire net_18;
  wire net_20;
  wire net_

In [25]:
import klayout.db as kdb


# 1. Read GDS.
layout = kdb.Layout()
layout.read("../puzzle.gds")
top = layout.top_cell()


# 2. Define the SKY130 conducting stack.
stack = [
    ("li",   67, 20),
    ("mcon", 67, 44),
    ("met1", 68, 20),
    ("via1", 68, 44),
    ("met2", 69, 20),
    ("via2", 69, 44),
    ("met3", 70, 20),
    ("via3", 70, 44),
    ("met4", 71, 20),
    ("via4", 71, 44),
    ("met5", 72, 20),
]


# 3. Extract physical conductor connectivity.
l2n = kdb.LayoutToNetlist(top.name, layout.dbu)
regions = {}

for name, layer, datatype in stack:
    layer_index = layout.layer(layer, datatype)
    region = kdb.Region(top.begin_shapes_rec(layer_index))

    l2n.register(region, name)
    l2n.connect(region)

    regions[name] = region


# Connect LI to mcon, mcon to M1, M1 to via1, and so on.
for lower, upper in zip(stack, stack[1:]):
    lower_name = lower[0]
    upper_name = upper[0]

    l2n.connect(
        regions[lower_name],
        regions[upper_name],
    )

l2n.extract_netlist()


# 4. Find exactly one placed logical standard cell.
li_pin_label_layer = layout.layer(67, 5)

power_pins = {
    "VGND",
    "VPWR",
    "VNB",
    "VPB",
}

logical_element = None

for instance_id, instance in enumerate(top.each_inst()):
    cell_type = instance.cell.name

    # Ignore via cells and other physical helper cells.
    if not cell_type.startswith("sky130_fd_sc_hd__"):
        continue

    pin_nets = {}

    # Pin names are text labels inside the standard-cell definition.
    for shape in instance.cell.shapes(li_pin_label_layer).each():
        if not shape.is_text():
            continue

        pin_name = shape.text.string

        if pin_name in power_pins:
            continue

        # Pin coordinates are local to the standard cell.
        displacement = shape.text.trans.disp
        local_point = kdb.Point(
            displacement.x,
            displacement.y,
        )

        # Convert local coordinates to top-level GDS coordinates.
        top_point = instance.trans * local_point

        # Determine which extracted physical net touches this pin.
        net = l2n.probe_net(
            regions["li"],
            top_point,
        )

        if net is not None:
            pin_nets.setdefault(pin_name, set()).add(
                net.cluster_id
            )

    # Ignore cells with no resolved signal pins.
    if not pin_nets:
        continue

    bbox = instance.cell.bbox().transformed(instance.trans)

    logical_element = {
        "instance_id": instance_id,
        "cell_type": cell_type,
        "transform": str(instance.trans),
        "bbox": (
            bbox.left,
            bbox.bottom,
            bbox.right,
            bbox.top,
        ),
        "pin_nets": {
            pin: sorted(nets)
            for pin, nets in pin_nets.items()
        },
    }

    # Stop after exactly one logical element.
    break


print(logical_element)

{'instance_id': 51, 'cell_type': 'sky130_fd_sc_hd__clkbuf_4', 'transform': 'r0 97520,70720', 'bbox': (97330, 70480, 100470, 73680), 'pin_nets': {'X': [683], 'A': [348]}}
